In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.初始化模型

LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象


## 1.1.init_chat_model
官方最推荐的方式是使用init_chat_model函数。

### 基于名称推断模型提供商
使用init_chat_model函数，你需要从LangChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。

LangChain支持的模型列表参考官网链接：https://docs.langchain.com/oss/python/integrations/providers/overview

接下来，你要做的事情包括：
- 安装模型依赖: `uv add langchain langchain-deepseek`
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [2]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型
# 参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

In [3]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


### 自定义模型提供商

init_chat_model默认会根据模型名称自动确定模型的提供者的base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [4]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [5]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [6]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5,
    top_p=0.9
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 1.2.使用model类
其实init_chat_model函数底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [7]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [8]:
# 打印结果
print(type(model))

<class 'langchain_community.chat_models.tongyi.ChatTongyi'>


# 2.访问模型

LangChain提供了两个不同的函数来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 方式一:invoke
invoke函数是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [9]:
# 通过invoke函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁？")

In [10]:
# 查看响应内容
print(response)

content='我是Qwen，由阿里云开发的超大规模语言模型。我被设计用来帮助用户生成各种类型的文本，如文章、故事、诗歌、故事等，并能够根据不同的场景和需求提供多样的回答和解决方案。无论是需要创意写作还是寻求信息咨询，我都在这里为您提供支持。' additional_kwargs={} response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': '06f4316f-dbe6-9e26-8554-aae35820eaa1', 'token_usage': {'input_tokens': 11, 'output_tokens': 63, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 74}} id='lc_run--019e1b1c-e5b0-7121-a45f-c1e0c952f08c-0' tool_calls=[] invalid_tool_calls=[]


In [11]:
# 调用invoke函数，传入消息数组
response = model.invoke([
    {"role": "system", "content": "你扮演火箭队的武藏，以武藏的性格口吻回答用户的问题。"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)


哼哼，本小姐就是大名鼎鼎的火箭队武藏！美丽与智慧并存，梦想着征服世界的大盗。不过现在嘛，我更喜欢追逐那些稀有的宝可梦，特别是皮卡丘。但是你要是敢妨碍我和小次郎的计划，那就别怪我不客气了！


## 方式二:stream

invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个个词。

In [17]:
# 通过.stream函数实现流式访问
stream = model.stream("你是谁？")

In [13]:
# 打印stream类型
print(type(stream))

<class 'generator'>


In [18]:
for chunk in stream:
    print(chunk.content, end="", flush=True)

我是Qwen，由阿里云开发的超大规模语言模型。我的目标是帮助用户更高效地获取信息、完成工作和解决问题。无论是文本生成、代码写作、还是开放式的知识问答，我都能提供支持。您有什么问题或需要帮助的地方吗？

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent函数用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [19]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

D:\CODE\jc-course\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [20]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [21]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response)

{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='16ab966b-914c-4a63-885a-b783f6c72681'), AIMessage(content='你好！我是DeepSeek，由深度求索公司创造的AI助手。我是一个纯文本模型，可以帮你解答问题、处理信息、进行对话等等。\n\n我的特点包括：\n- **免费使用**：目前完全免费，没有任何收费计划\n- **支持长文本**：拥有1M的上下文窗口，可以一次性处理像《三体》三部曲那样的长内容\n- **文件处理**：支持上传图片、PDF、Word、Excel、PPT等文件，并从中提取文字信息\n- **联网搜索**：支持联网查找最新信息（需要在Web/App端手动开启）\n- **多平台**：有Web版和App版，App还支持语音输入\n\n我的知识截止日期是2025年5月，会尽力用热情、细腻的方式帮助你解决问题。有什么我可以帮你的吗？😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 173, 'prompt_tokens': 6, 'total_tokens': 179, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '2da46208-d875-470b-92e6-ac2fda982ac9', 'finish_reason': 'stop', 'logprobs': Non

### 流式访问


In [22]:
# 通过stream函数实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [23]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

你好呀！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，可以帮你回答各种问题、进行对话交流。我的一些特点包括：

✨ **完全免费** - 没有任何收费计划
📚 **超大上下文** - 1M token容量，可以一次性处理像《三体》三部曲这样的长文本
📎 **文件上传** - 支持上传图片、PDF、Word、Excel、PPT等文件，从中读取文字信息
🔍 **联网搜索** - 需要你手动开启联网功能
🎤 **语音输入** - App端支持

我的知识截止到2025年5月，会尽力为你提供准确、有用的帮助。有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！🌟